## Setup: Mount Drive and Install Dependencies
This first setup step mounts Google Drive so generated audio and comparison outputs can be written to persistent storage outside the Colab runtime. It also installs the external audio and machine-learning packages used later in the notebook, including HuggingFace dataset access, torchaudio, librosa, and soundfile. Running this cell first ensures the rest of the pipeline can load data, download model weights, and save reconstruction artifacts.


In [ ]:
# CELL 1 - Mount Google Drive and install packages
from google.colab import drive
drive.mount("/content/drive")

%pip install -q datasets huggingface_hub librosa torchaudio soundfile pyyaml


## Imports
This section loads the standard Python modules used for paths, JSON metadata, math utilities, and collections. It also imports PyTorch, torchaudio, NumPy, librosa, soundfile, and plotting tools for model inference, audio conversion, and visualisation. The notebook adds the repository root to `sys.path` so the local `models/` and `utils/` packages can be imported directly.


In [ ]:
# CELL 2 - Imports
import os
import sys
import json
import math
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np

import librosa
import librosa.display
import librosa.feature.inverse
import matplotlib.pyplot as plt
import soundfile as sf

from datasets import load_dataset
from huggingface_hub import hf_hub_download

from IPython.display import Audio, display
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models").is_dir() and (PROJECT_ROOT.parent / "models").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.cnn_ae import CNNAutoencoder
from models.unet import UNet
from models.vae import VAE
from models.base_diffusion import DiffusionSchedule, DiffusionU_Net
from utils.mel2wav import Mel2Waveform


## Global Configuration
The global configuration chooses the active compute device, using CUDA when a GPU is available and CPU otherwise. It defines the HuggingFace repository IDs used for the processed dataset and model checkpoints. The same section sets the audio and mel-spectrogram parameters, including sample rate, FFT size, hop length, window length, and mel-bin count, so inference and Griffin-Lim inversion use consistent settings.


In [ ]:
# CELL 3 - Global config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HF_DATASET_REPO  = "han2o/grant-ortsaem-processedV3"
HF_WEIGHTS_REPO  = "han2o/inpaint_diffusion"
VARIANT          = "short_gaps"   # change to "long_gaps" if needed
GDRIVE_OUT       = Path("/content/drive/MyDrive/music_inpainting_project/reverse_pipeline")
GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
GRIFFINLIM_ITERS = 128

SAMPLE_RATE = 16000
N_FFT       = 1024
HOP_LENGTH  = 256
WIN_LENGTH  = 1024
N_MELS      = 128
F_MIN       = 30.0
F_MAX       = 8000.0
POWER       = 2.0

MEL_CONFIG_PATH = PROJECT_ROOT / "data" / "configs" / "data_configs.yaml"

print("device:", device)
print("outputs:", GDRIVE_OUT)


## DataLoader
This section wraps the HuggingFace streaming `IterableDataset` in a PyTorch dataloader. The wrapper converts spectrogram, target, and mask arrays into tensors while preserving metadata fields such as `recording_idx`, `clip_idx`, and frame boundaries. Keeping those keys attached to each batch is essential because later cells use them to match parquet samples to their corresponding debug WAV files on HuggingFace Hub.


In [ ]:
# CELL 4 - DataLoader factory function
from torch.utils.data import IterableDataset, DataLoader

META_KEYS = [
    "example_id", "split", "recording_idx", "clip_idx", "audio_filename",
    "sample_rate", "clip_duration_seconds", "n_mels", "time_frames",
    "hop_length", "gap_seconds", "gap_frames", "mask_start_frame", "mask_end_frame",
]

class ReversePipelineIterableDataset(IterableDataset):
    def __init__(self, hf_iterable):
        self.hf_iterable = hf_iterable

    @staticmethod
    def _as_spec_tensor(value):
        arr = np.asarray(value, dtype=np.float32)
        if arr.ndim == 2:
            arr = arr[None, :, :]
        return torch.tensor(arr, dtype=torch.float32)

    def __iter__(self):
        for row in self.hf_iterable:
            item = {
                "x": self._as_spec_tensor(row["masked_spectrogram"]),
                "y": self._as_spec_tensor(row["spectrogram"]),
                "mask": self._as_spec_tensor(row["mask"]),
            }
            for key in META_KEYS:
                if key in row:
                    item[key] = row[key]
            yield item


def _make_loader(split):
    data_files = f"hf://datasets/{HF_DATASET_REPO}/{VARIANT}/{split}-*.parquet"
    hf_split = load_dataset("parquet", data_files=data_files, split="train", streaming=True)
    return DataLoader(ReversePipelineIterableDataset(hf_split), batch_size=4, num_workers=0)


def make_test_loader():
    return _make_loader("test")


def make_train_loader():
    return _make_loader("train")


## Model Definitions
The notebook uses five model families: CNN AutoEncoder, U-Net, VAE, Base Diffusion, and Retrieval-Conditioned Diffusion. The CNN AE, U-Net, VAE, and Base Diffusion implementations are imported from the repository's `models/` package to keep this inference notebook aligned with the modular code. The Retrieval-Conditioned Diffusion model remains defined inline because it has not yet been modularised into `models/`.


In [ ]:
# CELL 5 - Shared model building blocks
# Shared model building blocks are provided by the imported repository modules.
# The retrieval-conditioned diffusion block below keeps its own helpers inline because
# that model is not yet modularised in models/.


In [ ]:
# CELL 6 - CNN AutoEncoder model definition
# CNNAutoencoder is imported from models.cnn_ae.


In [ ]:
# CELL 7 - U-Net model definition
# UNet is imported from models.unet.


In [ ]:
# CELL 8 - VAE model definition (shared by both diffusion models)
# VAE is imported from models.vae.


In [ ]:
# CELL 9 - Base Diffusion U-Net model definition and inference
# DiffusionU_Net and DiffusionSchedule are imported from models.base_diffusion.

@torch.no_grad()
def infer_base_diffusion(vae, unet, schedule, x_masked, mask, device, num_steps=200, use_self_conditioning=True):
    """
    Returns raw predicted spectrogram [B, 1, 128, T] WITHOUT merging with input.
    The caller (reconstruct_from_debug) performs the merge.
    """
    out = unet.infer_diffusion_latent(
        vae=vae,
        schedule=schedule,
        x_masked=x_masked,
        mask=mask,
        device=device,
        num_steps=num_steps,
        add_noise=True,
        return_all_steps=False,
        use_self_conditioning=use_self_conditioning,
    )
    return out["x_hat"]


In [ ]:
# CELL 10 - Retrieval-Conditioned Diffusion model definition and inference
# The retrieval classes below are copied from notebooks/ret_diffusion.ipynb so
# the inference notebook matches the trained retrieval checkpoint architecture.

# helper utilities function

# groupnorm function due to small batch size
def make_norm(channels, max_groups=8):
    groups = min(max_groups, channels)
    while channels % groups != 0 and groups > 1:
        groups -= 1
    return nn.GroupNorm(groups, channels)

# get learning rate for scheduling and resume training
def get_lr(optimiser):
    return optimiser.param_groups[0]["lr"]

# computes fraction
def compute_gap_ratio(mask):
    # fraction of missing region in the sample
    return mask.float().mean(dim=(1, 2, 3), keepdim=True)

# extracts schedule values at timestep t and reshapes for broadcatsing
def extract(a, t, x_shape):
    b = t.shape[0]
    out = a.gather(0, t)
    reshape_dims = (b,) + (1,) * (len(x_shape) - 1)
    return out.view(*reshape_dims)


def q_sample(z_0, t, noise, schedule):
    return extract(schedule.sqrt_alpha_bars, t, z_0.shape) * z_0 + extract(schedule.sqrt_one_minus_alpha_bars, t, z_0.shape) * noise


def predict_x0(z_t, eps_hat, t, schedule):
    return (z_t - extract(schedule.sqrt_one_minus_alpha_bars, t, z_t.shape) * eps_hat) / (extract(schedule.sqrt_alpha_bars, t, z_t.shape) + 1e-8)


def preserve_known_region(z_t, z_known, mask_latent):
    return z_t * mask_latent + z_known * (1.0 - mask_latent)

# padding as defined earlier
def padding(x, multiple=8):
    _, _, h, w = x.shape
    pad_h = (multiple - (h % multiple)) % multiple
    pad_w = (multiple - (w % multiple)) % multiple

    x_pad = F.pad(x, (0, pad_w, 0, pad_h), mode="constant", value=0.0)

    pad_info = {
        "orig_h": h,
        "orig_w": w,
        "pad_h": pad_h,
        "pad_w": pad_w,
    }
    return x_pad, pad_info

# unpadding
def unpadding(x, pad_info):
    return x[..., :pad_info["orig_h"], :pad_info["orig_w"]]

@torch.no_grad()
def encode2latentmean(vae, x):
    mu, logvar = vae.encode(x)
    return mu

@torch.no_grad()
def decode_from_latent(vae, z):
    return vae.decode(z)

# find start and end column indices of the mask region
def find_gap_from_mask(mask_latent):
    B, _, H, W = mask_latent.shape

    # average across freq dim and threshold
    gap_1d = (mask_latent.mean(dim=2) > 0.5).squeeze(1)

    start_idx = torch.zeros(B, dtype=torch.long, device=mask_latent.device)
    end_idx = torch.full((B,), W, dtype=torch.long, device=mask_latent.device)

    for b in range(B):
        idx = torch.where(gap_1d[b])[0]
        if len(idx) > 0:
            start_idx[b] = idx[0]
            end_idx[b] = idx[-1] + 1

    return start_idx, end_idx

# extract gap from local window
def extract_gap_window(z_context, m_latent, window_size=64):

    B, C, H, W = z_context.shape
    gap_start, gap_end = find_gap_from_mask(m_latent)
    # invert gap mask to get mask over known regions
    known_mask = 1.0 - m_latent

    z_left_list, z_right_list = [], []
    m_left_list, m_right_list = [], []

    for b in range(B):
        s = gap_start[b].item()
        e = gap_end[b].item()

        # clamp window boundaries to valid column range
        left_start = max(0, s - window_size)
        left_end = s
        right_start = e
        right_end = min(W, e + window_size)

        # slice the left and right ocntext windows form z and mask
        z_left = z_context[b:b+1, :, :, left_start:left_end]
        z_right = z_context[b:b+1, :, :, right_start:right_end]
        m_left = known_mask[b:b+1, :, :, left_start:left_end]
        m_right = known_mask[b:b+1, :, :, right_start:right_end]

        # pad left window on the left side if the gap is too close to the start
        if z_left.shape[-1] < window_size:
            pad = window_size - z_left.shape[-1]
            z_left = F.pad(z_left, (pad, 0, 0, 0), value=0.0)
            m_left = F.pad(m_left, (pad, 0, 0, 0), value=0.0)

        # pad right window on the right if the gap is too close to the end
        if z_right.shape[-1] < window_size:
            pad = window_size - z_right.shape[-1]
            z_right = F.pad(z_right, (0, pad, 0, 0), value=0.0)
            m_right = F.pad(m_right, (0, pad, 0, 0), value=0.0)

        z_left_list.append(z_left)
        z_right_list.append(z_right)
        m_left_list.append(m_left)
        m_right_list.append(m_right)

    # stack all samples back into batch tensors
    z_left = torch.cat(z_left_list, dim=0)
    z_right = torch.cat(z_right_list, dim=0)
    m_left = torch.cat(m_left_list, dim=0)
    m_right = torch.cat(m_right_list, dim=0)

    return z_left, z_right, m_left, m_right


# build normalised retrieval embedding from latent context surrounding gap region
def build_retrieval_emb(z_context, m_latent, window_size=64, pooled_hw=(8, 8)):
    """
    embedding captures local structure on boths sides of the gap by
      1. extracting fixed-size left/right context windows in latent space
      2. masking each window to retain only known (non-gap) columns
      3. spatially pooling each window to a fixed resolution
      4. concatenating and L2-normalising into a single query vector
    """
    z_left, z_right, m_left, m_right = extract_gap_window(
        z_context=z_context,
        m_latent=m_latent,
        window_size=window_size,
    )

    z_left = z_left * m_left
    z_right = z_right * m_right

    left_pool = F.adaptive_avg_pool2d(z_left, pooled_hw)
    right_pool = F.adaptive_avg_pool2d(z_right, pooled_hw)

    emb = torch.cat([
        left_pool.flatten(1),
        right_pool.flatten(1),
    ], dim=1)

    emb = F.normalize(emb, dim=1)
    return emb

# convservative duplicate checks
def same_source_clip(meta_a, meta_b):
    """
    prefer example_id when available.
    fall back to clip/source identifiers.
    """
    if meta_a is None or meta_b is None:
        return False

    if "example_id" in meta_a and "example_id" in meta_b:
        if meta_a["example_id"] == meta_b["example_id"]:
            return True

    keys = ["audio_filename", "recording_idx", "clip_idx"]
    if all(k in meta_a for k in keys) and all(k in meta_b for k in keys):
        return (
            meta_a["audio_filename"] == meta_b["audio_filename"]
            and meta_a["recording_idx"] == meta_b["recording_idx"]
            and meta_a["clip_idx"] == meta_b["clip_idx"]
        )

    return False

# extract per sample metadata from data loader batch
def batch_query_meta_from_batch(batch, batch_size):
    metas = []
    for i in range(batch_size):
        meta = {}
        for key in ["example_id", "audio_filename", "recording_idx", "clip_idx", "gap_seconds"]:
            if key in batch:
                value = batch[key]
                if torch.is_tensor(value):
                    try:
                        meta[key] = value[i].item()
                    except Exception:
                        meta[key] = value[i]
                else:
                    meta[key] = value[i]
        metas.append(meta)
    return metas

# retrieval bank
class RetrievalBank:
    def __init__(self, device="cpu"):
        self.device = device
        self.bank_embeds = None      # embeddings used for search
        self.bank_contexts = None    # masked known-context latents
        self.bank_targets = None     # full clean latent targets
        self.bank_masks = None
        self.bank_meta = None

    @torch.no_grad()
    def build(
        self,
        vae,
        dataloader,
        device,
        max_items=4000,
        desc="building retrieval bank",
        window_size=64,
        pooled_hw=(8, 8),
    ):
        embeds = []
        contexts = []
        targets = []
        masks = []
        metas = []

        vae.eval()
        n_total = 0

        for batch in tqdm(dataloader, desc=desc):
            x = batch["x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)
            m = batch["mask"].to(device, non_blocking=True)

            x, _ = padding(x, multiple=8)
            y, _ = padding(y, multiple=8)
            m, _ = padding(m, multiple=8)

            z_masked = encode2latentmean(vae, x)
            z_clean = encode2latentmean(vae, y)

            m_latent = F.interpolate(m, size=z_masked.shape[-2:], mode="nearest")
            z_context = z_masked * (1.0 - m_latent)

            emb = build_retrieval_emb(
                z_context=z_context,
                m_latent=m_latent,
                window_size=window_size,
                pooled_hw=pooled_hw,
            )

            embeds.append(emb.cpu())
            contexts.append(z_context.cpu())
            targets.append(z_clean.cpu())
            masks.append(m_latent.cpu())

            batch_size = x.shape[0]
            for i in range(batch_size):
                meta = {}
                for key in ["example_id", "audio_filename", "recording_idx", "clip_idx", "gap_seconds"]:
                    if key in batch:
                        value = batch[key]
                        if torch.is_tensor(value):
                            try:
                                meta[key] = value[i].item()
                            except Exception:
                                meta[key] = value[i]
                        else:
                            meta[key] = value[i]
                metas.append(meta)

            n_total += batch_size
            if n_total >= max_items:
                break

        self.bank_embeds = torch.cat(embeds, dim=0)[:max_items].contiguous()
        self.bank_contexts = torch.cat(contexts, dim=0)[:max_items].contiguous()
        self.bank_targets = torch.cat(targets, dim=0)[:max_items].contiguous()
        self.bank_masks = torch.cat(masks, dim=0)[:max_items].contiguous()
        self.bank_meta = metas[:max_items]

        print(f"built retrieval bank with {self.bank_embeds.shape[0]} items")

    @torch.no_grad()
    def query(self, query_context_latent, query_mask_latent, top_k=5, query_meta=None):
        assert self.bank_embeds is not None, "build the retrieval bank first"

        q = build_retrieval_emb(
            z_context=query_context_latent,
            m_latent=query_mask_latent,
        )

        sims = torch.matmul(q.cpu(), self.bank_embeds.T)  # [b, n]

        bsz, n_items = sims.shape
        overfetch_k = min(max(top_k * 8, top_k + 10), n_items)

        raw_scores, raw_idx = torch.topk(sims, k=overfetch_k, dim=1)

        final_scores = []
        final_idx = []

        for b in range(bsz):
            kept_scores = []
            kept_indices = []

            q_meta = query_meta[b] if query_meta is not None else None

            for score, idx in zip(raw_scores[b].tolist(), raw_idx[b].tolist()):
                bank_meta = self.bank_meta[idx] if self.bank_meta is not None else None

                # skip exact self / same source clip when metadata is available
                if q_meta is not None and bank_meta is not None:
                    if same_source_clip(q_meta, bank_meta):
                        continue

                kept_scores.append(score)
                kept_indices.append(idx)

                if len(kept_indices) >= top_k:
                    break

            # fallback if filtering removed too many results
            if len(kept_indices) == 0:
                kept_scores.append(raw_scores[b, 0].item())
                kept_indices.append(raw_idx[b, 0].item())

            while len(kept_indices) < top_k:
                kept_scores.append(0.0)
                kept_indices.append(kept_indices[-1])

            final_scores.append(torch.tensor(kept_scores, dtype=torch.float32))
            final_idx.append(torch.tensor(kept_indices, dtype=torch.long))

        top_scores = torch.stack(final_scores, dim=0)
        top_idx = torch.stack(final_idx, dim=0)

        retrieved_top1_target = self.bank_targets[top_idx[:, 0]].to(query_context_latent.device)
        sim_top1 = top_scores[:, 0].to(query_context_latent.device).unsqueeze(1)

        return {
            "retrieved_top1_target": retrieved_top1_target,
            "sim_top1": sim_top1,
            "top_scores": top_scores,
            "top_idx": top_idx,
        }

@torch.no_grad()
def heuristic_selective_mix(
    candidate_targets,
    candidate_scores,
    candidate_valid_mask=None,
    temperature=0.08,
    sharpen_power=2.0,
    min_keep_weight=0.05
    ):

    # added for safety
    scores = candidate_scores.float()

    # mask invalid padded slots before softmax
    if candidate_valid_mask is not None:
        scores = torch.where(
            candidate_valid_mask > 0,
            scores,
            torch.full_like(scores, -1e4)
        )

    weights = torch.softmax(candidate_scores / temperature, dim=1)
    weights = weights ** sharpen_power
    weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)

    if min_keep_weight is not None and min_keep_weight > 0:
        keep_mask = weights >= min_keep_weight

        if candidate_valid_mask is not None:
            keep_mask = keep_mask & (candidate_valid_mask > 0)

        weights = torch.where(
            keep_mask,
            weights,
            torch.zeros_like(weights),
        )

        # if everything was dropped, fall back to top-1 valid candidate
        row_sum = weights.sum(dim=1, keepdim=True)
        zero_rows = row_sum.squeeze(1) <= 1e-8

        if zero_rows.any():
            fallback_idx = torch.argmax(scores[zero_rows], dim=1)
            fallback = torch.zeros_like(weights[zero_rows])
            fallback.scatter_(1, fallback_idx.unsqueeze(1), 1.0)
            weights[zero_rows] = fallback

        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)

    fused = (
        weights[:, :, None, None, None].to(candidate_targets.dtype) * candidate_targets
    ).sum(dim=1)

    return {
        "fused_target": fused,
        "weights": weights,
    }


@torch.no_grad()
def sample_retrieval_strength(
    batch_size,
    device,
    p_none=0.20,
    p_weak=0.40,
    p_mid=0.20,
    p_strong=0.20,
    weak_range=(0.30, 0.50),
    mid_range=(0.55, 0.75),
    strong_range=(0.85, 1.00),
):
    # sample retrieval strength per sample
    # this is the v1/v2/v3 training trick that teaches the model
    # how to use no retrieval, weak retrieval, and stronger retrieval

    u = torch.rand(batch_size, device=device)
    r = torch.zeros(batch_size, device=device)

    weak_mask = (u >= p_none) & (u < p_none + p_weak)
    mid_mask = (u >= p_none + p_weak) & (u < p_none + p_weak + p_mid)
    strong_mask = u >= (p_none + p_weak + p_mid)

    if weak_mask.any():
        r[weak_mask] = weak_range[0] + (weak_range[1] - weak_range[0]) * torch.rand(weak_mask.sum(), device=device)

    if mid_mask.any():
        r[mid_mask] = mid_range[0] + (mid_range[1] - mid_range[0]) * torch.rand(mid_mask.sum(), device=device)

    if strong_mask.any():
        r[strong_mask] = strong_range[0] + (strong_range[1] - strong_range[0]) * torch.rand(strong_mask.sum(), device=device)

    return r.view(batch_size, 1, 1, 1)



class LearnedFusionModule(nn.Module):
    """
    a model to learn which of the topk candidates matters more,
    and how much to weight them
    """
    def __init__(self, latent_channels=8, hidden_dim=128):
        super().__init__()

        # input = pooled context + pooled candidate + retrieval score + gap ratio
        # in_dim = latent_channels + latent_channels + 1 + 1

        # input_dim =
            # - global pooled query context        -> latent_channels
            # - gap pooled candidate              -> latent_channels
            # - global pooled candidate           -> latent_channels
            # - retrieval score                   -> 1
            # - gap ratio                         -> 1
        in_dim = latent_channels * 3 + 2

        self.score_net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    # added for gap pooling instead of global
    def global_pool(self, x):
        return F.adaptive_avg_pool2d(x, output_size=1).flatten(1)

    # added for gap polling instead of global
    def gap_pool(self, x, m_latent):
        # m_latent: [b, 1, h, w], 1 inside the gap
        gap_mask = m_latent.expand(-1, x.shape[1], -1, -1).to(x.dtype)

        gap_sum = (x * gap_mask).sum(dim=(-2, -1))
        gap_count = gap_mask.sum(dim=(-2, -1)).clamp(min=1.0)

        return gap_sum / gap_count

    def forward(self, candidate_targets, candidate_scores, candidate_valid_mask, z_context, m_latent):
        # candidate_targets: [b, k, c, h, w]
        # candidate_scores:  [b, k]

        bsz, k, c, h, w = candidate_targets.shape

        # old
        # # pooled context summary
        # context_vec = F.adaptive_avg_pool2d(z_context, output_size=1).flatten(1)
        # # gap ratio summary
        # gap_ratio = compute_gap_ratio(m_latent).view(bsz, 1)

        # new
        # pooled context summary
        context_global_vec = self.global_pool(z_context)
        # gap ratio summary
        gap_ratio = compute_gap_ratio(m_latent).view(bsz, 1).to(z_context.dtype)

        # logits = []
        # for i in range(k):
        #     cand_vec = F.adaptive_avg_pool2d(candidate_targets[:, i], output_size=1).flatten(1)
        #     score_i = candidate_scores[:, i:i+1]

        #     feat = torch.cat([context_vec, cand_vec, score_i, gap_ratio], dim=1)
        #     logits_i = self.score_net(feat)
        #     logits.append(logits_i)

        # logits = torch.cat(logits, dim=1).float()

        # # mask invalid padded candidates
        # logits = torch.where(
        #     candidate_valid_mask > 0,
        #     logits,
        #     torch.full_like(logits, -1e4)
        # )

        logits = []
        for i in range(k):
            cand = candidate_targets[:, i]
            score_i = candidate_scores[:, i:i+1].to(z_context.dtype)

            cand_gap_vec = self.gap_pool(cand, m_latent)
            cand_global_vec = self.global_pool(cand)

            feat = torch.cat(
                [context_global_vec, cand_gap_vec, cand_global_vec, score_i, gap_ratio],
                dim=1
            )

            logits_i = self.score_net(feat)
            logits.append(logits_i)

        logits = torch.cat(logits, dim=1).float()

        # invalid padded candidates are masked
        logits = torch.where(
            candidate_valid_mask > 0,
            logits,
            torch.full_like(logits, -1e4)
        )

        weights = torch.softmax(logits, dim=1)

        # fused = (weights[:, :, None, None, None] * candidate_targets.float()).sum(dim=1)
        fused = (
            weights[:, :, None, None, None].to(candidate_targets.dtype) * candidate_targets
        ).sum(dim=1)

        return {
            "fused_target": fused,
            "weights": weights,
            "logits": logits
        }

# encodes scalar diffusion timestep t into a sinusoidal embedding vector.
class SinusoidalTimeEmb(nn.Module):
    """
    same positional encoding scheme as the original transformer paper,
    but adapted for diffusion timesteps. gives the UNet a continuous,
    frequency rich representation of how noisy the input currently is.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        freq_factor = math.log(10000) / max(half_dim - 1, 1)
        freqs = torch.exp(
            torch.arange(half_dim, device=t.device, dtype=torch.float32) * (-freq_factor)
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)

        # pad if dim is odd
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb

# projects sinusoidal time embedding through a small MLP to increase expressiveness
class TimeEmbMLP(nn.Module):
    def __init__(self, time_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )

    def forward(self, t_emb):
        return self.net(t_emb)

# SE block
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1)

    def forward(self, x):
        scale = self.pool(x)
        scale = F.silu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale))
        return x * scale

# downsmapling
class DownSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)

# upsampling block
class UpSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)

# core residual blokc for diffusion unet
class DiffusionResBlock(nn.Module):
    """
    applies two conv layers with GroupNorm and SiLU activations.

    time conditioning is injected using FiLM:
    -  the time embedding is projected to per-channel scale and shift values that modulate the intermediate feature map after the first conv.
    - SE block recalibrates channel responses after the second conv.
    - 1x1 skip connection handles channel dimension changes.
    """
    def __init__(self, in_channels, out_channels, time_emb_dim, use_se=True):
        super().__init__()

        self.norm1 = make_norm(in_channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        # projects time embedding to scale + shift for FiLM conditioning
        self.time_proj = nn.Linear(time_emb_dim, out_channels * 2)

        self.norm2 = make_norm(out_channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.se = SEBlock(out_channels) if use_se else nn.Identity()
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        residual = self.skip(x)

        h = self.norm1(x)
        h = self.act1(h)
        h = self.conv1(h)

        film = self.time_proj(t_emb)
        scale, shift = torch.chunk(film, 2, dim=1)
        scale = scale[:, :, None, None]
        shift = shift[:, :, None, None]

        h = self.norm2(h)
        h = h * (1.0 + scale) + shift
        h = self.act2(h)
        h = self.conv2(h)
        h = self.se(h)

        return h + residual

# residual blokc with dilated convolution in the first layer
class DilatedResBlock(nn.Module):
    """
    increases the receptive field without adding parameters or reducing resolution.
    """
    def __init__(self, channels, dilation=2):
        super().__init__()
        self.norm1 = make_norm(channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(
            channels, channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation
        )

        self.norm2 = make_norm(channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = x

        h = self.norm1(x)
        h = self.act1(h)
        h = self.conv1(h)

        h = self.norm2(h)
        h = self.act2(h)
        h = self.conv2(h)

        return h + residual

# multi-head self-attention over 2D feature maps.
class SelfAttention2D(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        assert channels % num_heads == 0

        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        self.norm = make_norm(channels)
        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_k = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_v = nn.Conv2d(channels, channels, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        b, c, h, w = x.shape
        residual = x

        x = self.norm(x)

        q = self.to_q(x).view(b, self.num_heads, self.head_dim, h * w).permute(0, 1, 3, 2)
        k = self.to_k(x).view(b, self.num_heads, self.head_dim, h * w)
        v = self.to_v(x).view(b, self.num_heads, self.head_dim, h * w).permute(0, 1, 3, 2)

        attn = torch.matmul(q, k) / math.sqrt(self.head_dim)
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).contiguous().view(b, c, h, w)
        out = self.proj(out)

        return out + residual

# cross-attention block where the bottleneck feature map attends to external condiitoning tokens (local context or retrieved exampels)
class CrossAttentionBlock(nn.Module):
    """
    query = spatial tokens from the bottleneck featuremap
    key/value = conditioning tokens from context or retrieval

    this allow diffusion model to be guided by additional infomraiton beynd latent and timestep
    such as surrounding spectrogram context or a retrieved reference example
    """
    def __init__(self, channels, cond_dim, num_heads=4):
        super().__init__()
        assert channels % num_heads == 0

        self.channels = channels
        self.cond_dim = cond_dim
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        self.norm = make_norm(channels)
        # queries from feature map
        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        # keys from conditioning tokens
        self.to_k = nn.Linear(cond_dim, channels)
        # values from conditioning tokens
        self.to_v = nn.Linear(cond_dim, channels)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x, cond_tokens):
        b, c, h, w = x.shape
        residual = x

        x_norm = self.norm(x)

        # queries and key values
        q = self.to_q(x_norm).view(b, self.num_heads, self.head_dim, h * w).permute(0, 1, 3, 2)
        k = self.to_k(cond_tokens).view(b, -1, self.num_heads, self.head_dim).permute(0, 2, 3, 1)
        v = self.to_v(cond_tokens).view(b, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        attn = torch.matmul(q, k) / math.sqrt(self.head_dim)
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).contiguous().view(b, c, h, w)
        out = self.proj(out)

        return out + residual

class RetrievalEncoder(nn.Module):
    def __init__(self, latent_channels, cond_dim=256, base_channels=64):
        super().__init__()

        self.in_conv = nn.Conv2d(latent_channels, base_channels, kernel_size=3, padding=1)

        self.block1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
        )

        self.down1 = DownSample(base_channels)

        self.block2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
            nn.Conv2d(base_channels * 2, base_channels * 2, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )

        self.proj = nn.Conv2d(base_channels * 2, cond_dim, kernel_size=1)

    def forward(self, z_retrieved):
        x = self.in_conv(z_retrieved)
        x = self.block1(x)
        x = self.down1(x)
        x = self.block2(x)
        x = self.proj(x)

        b, d, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2).contiguous()  # [b, n, d]
        return tokens

class RetrievalConditionedDiffusionUNet(nn.Module):
    def __init__(
        self,
        latent_channels=8,
        base_channels=128,
        time_dim=256,
        cond_dim=256,
        retrieval_base_channels=64,
        num_retrieval_cross_attn=2,   # 1 = cross attention block, or 2
        fusion_mode="selective_mix",  # "selective_mix" or "learned_fusion"
    ):
        """
        selective_mix (hand designed rule):
          - use retrieved topk candidates, combines them using:
              1. use retrieval scores
              2. apply temp
              3. sharpen the weights
              4. remove tiny weights
              5. average with those weights

        learned_fusion (model selection):
          - let a model learn hwo to combine the topk retrievals
        """
        super().__init__()

        assert fusion_mode in ["selective_mix", "learned_fusion"]

        self.latent_channels = latent_channels
        self.time_dim = time_dim
        self.cond_dim = cond_dim
        self.num_retrieval_cross_attn = num_retrieval_cross_attn
        self.fusion_mode = fusion_mode

        # retrieval modules
        self.retrieval_encoder = RetrievalEncoder(
            latent_channels=latent_channels,
            cond_dim=cond_dim,
            base_channels=retrieval_base_channels,
        )

        if fusion_mode == "learned_fusion":
            self.fusion_module = LearnedFusionModule(
                latent_channels=latent_channels,
                hidden_dim=128
            )
        else:
            self.fusion_module = None

        # NEW: add learned null token for no-retrieval conditioning
        self.null_cond_token = nn.Parameter(torch.zeros(1, 1, cond_dim))

        # standard diffusion input stays unchanged
        in_channels = latent_channels + latent_channels + latent_channels + 1
        out_channels = latent_channels

        self.time_embed = SinusoidalTimeEmb(time_dim)
        self.time_mlp = TimeEmbMLP(time_dim)

        self.in_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        # encoder
        self.down1_block1 = DiffusionResBlock(base_channels, base_channels, time_emb_dim=time_dim, use_se=True)
        self.down1_block2 = DiffusionResBlock(base_channels, base_channels, time_emb_dim=time_dim, use_se=True)
        self.down1 = DownSample(base_channels)

        self.down2_block1 = DiffusionResBlock(base_channels, base_channels * 2, time_emb_dim=time_dim, use_se=True)
        self.down2_block2 = DiffusionResBlock(base_channels * 2, base_channels * 2, time_emb_dim=time_dim, use_se=True)
        self.down2 = DownSample(base_channels * 2)

        self.down3_block1 = DiffusionResBlock(base_channels * 2, base_channels * 4, time_emb_dim=time_dim, use_se=True)
        self.down3_block2 = DiffusionResBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim, use_se=True)
        self.down3 = DownSample(base_channels * 4)

        # bottleneck
        self.mid_block1 = DiffusionResBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim, use_se=True)
        self.mid_dilate1 = DilatedResBlock(base_channels * 4, dilation=2)
        self.mid_self_attn = SelfAttention2D(base_channels * 4, num_heads=4)
        self.mid_retrieval_attn = CrossAttentionBlock(base_channels * 4, cond_dim=cond_dim, num_heads=4)
        self.mid_dilate2 = DilatedResBlock(base_channels * 4, dilation=4)
        self.mid_block2 = DiffusionResBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim, use_se=True)

        # decoder
        self.up3 = UpSample(base_channels * 4)
        self.up3_block1 = DiffusionResBlock(base_channels * 8, base_channels * 4, time_emb_dim=time_dim, use_se=True)

        # this is the optional second retrieval attention used by v2/v3
        if self.num_retrieval_cross_attn >= 2:
            self.up3_retrieval_attn = CrossAttentionBlock(base_channels * 4, cond_dim=cond_dim, num_heads=4)
        else:
            self.up3_retrieval_attn = None

        self.up3_block2 = DiffusionResBlock(base_channels * 4, base_channels * 2, time_emb_dim=time_dim, use_se=True)

        self.up2 = UpSample(base_channels * 2)
        self.up2_block1 = DiffusionResBlock(base_channels * 4, base_channels * 2, time_emb_dim=time_dim, use_se=True)
        self.up2_block2 = DiffusionResBlock(base_channels * 2, base_channels, time_emb_dim=time_dim, use_se=True)

        self.up1 = UpSample(base_channels)
        self.up1_block1 = DiffusionResBlock(base_channels * 2, base_channels, time_emb_dim=time_dim, use_se=True)
        self.up1_block2 = DiffusionResBlock(base_channels, base_channels, time_emb_dim=time_dim, use_se=True)

        self.out_norm = make_norm(base_channels)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=3, padding=1)

    def match_spatial(self, x, ref):
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:], mode="nearest")
        return x

    def build_retrieval_condition(
        self,
        candidate_targets,
        candidate_scores,
        candidate_valid_mask,
        z_context,
        m_latent,
        retrieval_strength,
        selective_temperature=0.08,
        selective_sharpen_power=2.0,
        selective_min_keep_weight=0.05,
    ):
        # choose how retrieved candidates are fused

        # fuse retrieval candidates according to the chosen version
        if self.fusion_mode == "learned_fusion":
            fusion_out = self.fusion_module(
                candidate_targets=candidate_targets,
                candidate_scores=candidate_scores,
                candidate_valid_mask=candidate_valid_mask,
                z_context=z_context,
                m_latent=m_latent,
            )
            fused_target = fusion_out["fused_target"]
            fusion_weights = fusion_out["weights"]
            fusion_logits = fusion_out["logits"]

        else:
            fusion_out = heuristic_selective_mix(
                candidate_targets=candidate_targets,
                candidate_scores=candidate_scores,
                candidate_valid_mask=candidate_valid_mask,
                temperature=selective_temperature,
                sharpen_power=selective_sharpen_power,
                min_keep_weight=selective_min_keep_weight,
            )
            fused_target = fusion_out["fused_target"]
            fusion_weights = fusion_out["weights"]
            fusion_logits = candidate_scores

        # scale retrieval strength before encoding into tokens
        scaled_target = fused_target * retrieval_strength

        # convert retrieval latent into conditioning tokens
        cond_tokens = self.retrieval_encoder(scaled_target)

        # blend with a learned null token so strength=0 means a clean no-retrieval signal
        strength = retrieval_strength.view(-1, 1, 1).to(cond_tokens.dtype)   # [b, 1, 1]
        null_tokens = self.null_cond_token.to(cond_tokens.dtype).expand(
            cond_tokens.shape[0],
            cond_tokens.shape[1],
            -1
        )

        cond_tokens = strength * cond_tokens + (1.0 - strength) * null_tokens

        return {
            "cond_tokens": cond_tokens,
            "fused_target": fused_target,
            "fusion_weights": fusion_weights,
            "fusion_logits": fusion_logits,
            "retrieval_strength": retrieval_strength,
        }

    def forward(self, x, t, cond_tokens=None):
        t_emb = self.time_embed(t)
        t_emb = self.time_mlp(t_emb)

        x0 = self.in_conv(x)

        d1 = self.down1_block1(x0, t_emb)
        d1 = self.down1_block2(d1, t_emb)
        x1 = self.down1(d1)

        d2 = self.down2_block1(x1, t_emb)
        d2 = self.down2_block2(d2, t_emb)
        x2 = self.down2(d2)

        d3 = self.down3_block1(x2, t_emb)
        d3 = self.down3_block2(d3, t_emb)
        x3 = self.down3(d3)

        h = self.mid_block1(x3, t_emb)
        h = self.mid_dilate1(h)
        h = self.mid_self_attn(h)

        if cond_tokens is not None:
            h = self.mid_retrieval_attn(h, cond_tokens)

        h = self.mid_dilate2(h)
        h = self.mid_block2(h, t_emb)

        h = self.up3(h)
        h = self.match_spatial(h, d3)
        h = torch.cat([h, d3], dim=1)
        h = self.up3_block1(h, t_emb)

        if self.up3_retrieval_attn is not None and cond_tokens is not None:
            h = self.up3_retrieval_attn(h, cond_tokens)

        h = self.up3_block2(h, t_emb)

        h = self.up2(h)
        h = self.match_spatial(h, d2)
        h = torch.cat([h, d2], dim=1)
        h = self.up2_block1(h, t_emb)
        h = self.up2_block2(h, t_emb)

        h = self.up1(h)
        h = self.match_spatial(h, d1)
        h = torch.cat([h, d1], dim=1)
        h = self.up1_block1(h, t_emb)
        h = self.up1_block2(h, t_emb)

        h = self.out_norm(h)
        h = self.out_act(h)
        out = self.out_conv(h)
        return out




def _ddpm_reverse_step(z_t, eps_hat, t, schedule):
    beta_t = extract(schedule.betas, t, z_t.shape)
    alpha_t = extract(schedule.alphas, t, z_t.shape)
    alpha_bar_t = extract(schedule.alpha_bars, t, z_t.shape)
    posterior_var_t = extract(schedule.posterior_variance, t, z_t.shape)
    t_prev = torch.clamp(t - 1, min=0)
    alpha_bar_prev_t = extract(schedule.alpha_bars, t_prev, z_t.shape)
    is_t0 = (t == 0).float().view(z_t.shape[0], 1, 1, 1)
    alpha_bar_prev_t = is_t0 * torch.ones_like(alpha_bar_prev_t) + (1.0 - is_t0) * alpha_bar_prev_t
    z0_hat = predict_x0(z_t, eps_hat, t, schedule).clamp(-4.0, 4.0)
    coef1 = beta_t * torch.sqrt(alpha_bar_prev_t) / (1.0 - alpha_bar_t + 1e-8)
    coef2 = torch.sqrt(alpha_t) * (1.0 - alpha_bar_prev_t) / (1.0 - alpha_bar_t + 1e-8)
    model_mean = coef1 * z0_hat + coef2 * z_t
    nonzero_mask = (t != 0).float().view(z_t.shape[0], 1, 1, 1)
    z_prev = model_mean + nonzero_mask * torch.sqrt(torch.clamp(posterior_var_t, min=1e-20)) * torch.randn_like(z_t)
    return z_prev, z0_hat

@torch.no_grad()
def _build_ret_cond_tokens(
    ret_unet,
    retrieval_bank,
    z_context,
    mask_latent,
    device,
    top_k=5,
    max_keep=3,
):
    query_out = retrieval_bank.query(
        query_context_latent=z_context,
        query_mask_latent=mask_latent,
        top_k=top_k,
        query_meta=None,
    )

    keep = min(max_keep, query_out["top_idx"].shape[1])
    top_idx = query_out["top_idx"][:, :keep]
    top_scores = query_out["top_scores"][:, :keep].to(device)

    candidate_targets = retrieval_bank.bank_targets[top_idx].to(device)
    candidate_valid_mask = torch.ones_like(top_scores, device=device)
    retrieval_strength = torch.ones(z_context.shape[0], 1, 1, 1, device=device)

    cond = ret_unet.build_retrieval_condition(
        candidate_targets=candidate_targets,
        candidate_scores=top_scores,
        candidate_valid_mask=candidate_valid_mask,
        z_context=z_context,
        m_latent=mask_latent,
        retrieval_strength=retrieval_strength,
        selective_temperature=0.08,
        selective_sharpen_power=2.0,
        selective_min_keep_weight=0.05,
    )
    return cond["cond_tokens"]


@torch.no_grad()
def infer_ret_diffusion(
    vae,
    ret_unet,
    retrieval_bank,
    schedule,
    x_masked,
    mask,
    device,
    num_steps=200,
    top_k=5,
    max_keep=3,
    use_self_conditioning=True,
):
    """
    Returns raw predicted spectrogram [B, 1, 128, T] WITHOUT merging with input.
    The caller (reconstruct_from_debug) performs the merge.
    """
    vae.eval(); ret_unet.eval()
    x_masked = x_masked.to(device); mask = mask.to(device)
    orig_h, orig_w = x_masked.shape[-2:]

    x_masked_pad, pad_info = padding(x_masked, multiple=8)
    mask_pad, _ = padding(mask, multiple=8)
    z_masked = encode2latentmean(vae, x_masked_pad)
    mask_latent = F.interpolate(mask_pad, size=z_masked.shape[-2:], mode="nearest")
    z_context = z_masked * (1.0 - mask_latent)

    cond_tokens = _build_ret_cond_tokens(
        ret_unet=ret_unet,
        retrieval_bank=retrieval_bank,
        z_context=z_context,
        mask_latent=mask_latent,
        device=device,
        top_k=top_k,
        max_keep=max_keep,
    )

    z_t = torch.randn_like(z_masked)
    t_init = torch.full((z_masked.shape[0],), num_steps - 1, device=device, dtype=torch.long)
    known_noise = torch.randn_like(z_masked)
    z_known_init = q_sample(z_masked, t_init, known_noise, schedule)
    z_t = preserve_known_region(z_t, z_known_init, mask_latent)
    z0_selfcond = torch.zeros_like(z_masked)

    for step in tqdm(reversed(range(num_steps)), total=num_steps, desc="Sampling Retrieval Diffusion", leave=False):
        t = torch.full((z_t.shape[0],), step, device=device, dtype=torch.long)
        sc = z0_selfcond if use_self_conditioning else torch.zeros_like(z0_selfcond)
        model_input = torch.cat([z_t, z_masked, sc, mask_latent], dim=1)
        eps_hat = ret_unet(model_input, t, cond_tokens)
        z_prev, z0_hat = _ddpm_reverse_step(z_t, eps_hat, t, schedule)

        if step > 0:
            t_prev = torch.full((z_t.shape[0],), step - 1, device=device, dtype=torch.long)
            z_known = q_sample(z_masked, t_prev, known_noise, schedule)
            z_t = preserve_known_region(z_prev, z_known, mask_latent)
        else:
            z_t = preserve_known_region(z0_hat, z_masked, mask_latent)

        if use_self_conditioning:
            z0_selfcond = z0_hat.detach()

    pred_pad = decode_from_latent(vae, z_t)
    pred = unpadding(pred_pad, pad_info)[:, :, :orig_h, :orig_w]
    return pred


## Load Model Weights
This section downloads model checkpoints from HuggingFace Hub and loads each checkpoint state dict into the corresponding model instance. The VAE is frozen after loading because both diffusion samplers use it as a latent encoder and decoder rather than as a trainable component. The retrieval-conditioned diffusion checkpoint is loaded from Drive, and a retrieval bank is built from the training stream for conditioning.


In [ ]:
# CELL 11 - Load model weights from HuggingFace
def _checkpoint_info(ckpt):
    epoch = ckpt.get("epoch", ckpt.get("best_epoch", "unknown"))
    best_score = ckpt.get("best_score", ckpt.get("best_metric", ckpt.get("score", "unknown")))
    return epoch, best_score


def _load_checkpoint_from_hf(model, filename, model_name, strict=True):
    path = hf_hub_download(repo_id=HF_WEIGHTS_REPO, filename=filename, repo_type="model")
    ckpt = torch.load(path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    model.load_state_dict(state, strict=strict)
    model.eval()
    epoch, best_score = _checkpoint_info(ckpt if isinstance(ckpt, dict) else {})
    print(f"{model_name}: loaded {filename} | epoch={epoch} | best_score={best_score}")
    return ckpt


# 11a. CNN-AE
cnn_ae = CNNAutoencoder(base_channels=12).to(device)
cnn_ae_ckpt = _load_checkpoint_from_hf(cnn_ae, "cnn_ae/best_model.pt", "CNN-AE")

# 11b. U-Net
unet_model = UNet(n_channels=2, n_classes=1).to(device)
unet_ckpt = _load_checkpoint_from_hf(unet_model, "unet/best_model.pt", "U-Net")

# 11c. VAE
vae = VAE(in_channels=1, base_channels=64, latent_channels=8).to(device)
vae_ckpt = _load_checkpoint_from_hf(vae, "vae/best_model.pt", "VAE")

# freeze - exactly as in ret_diffusion notebook
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

latent_channels = vae.latent_channels  # used by both diffusion models and retrieval bank
print("Frozen VAE latent channels:", latent_channels)

# 11d. Base Diffusion
# The schedule is shared by the base and retrieval diffusion samplers.
diff_schedule = DiffusionU_Net.cosine_schedule(num_steps=1000, device=device)
diff_unet = DiffusionU_Net(
    latent_channels=latent_channels,
    base_channels=128,
    time_dim=256,
    num_steps=1000,
    schedule_device=device,
).to(device)
diff_ckpt = _load_checkpoint_from_hf(diff_unet, "base_diffusion/best_model.pt", "Base Diffusion")

# 11e. Retrieval-Conditioned Diffusion
ret_unet = RetrievalConditionedDiffusionUNet(
    latent_channels=8, base_channels=128, time_dim=256, cond_dim=256,
    retrieval_base_channels=64, num_retrieval_cross_attn=2,
    fusion_mode="selective_mix",
).to(device)

# load from Google Drive
ret_ckpt_path = "/content/drive/MyDrive/music_inpainting_project/enhanced_diffusion2/diffusion_retrieval_/checkpoints/best_model.pt"
checkpoint = torch.load(ret_ckpt_path, map_location=device)
state_dict = checkpoint["model_state_dict"]
model_keys = ret_unet.state_dict().keys()

# Adaptive compatibility for the only known checkpoint naming drift:
# some retrieval definitions call the self-attention projection "proj",
# while the shared base attention block calls the same layer "proj_out".
if (
    "mid_self_attn.proj.weight" in state_dict
    and "mid_self_attn.proj_out.weight" in model_keys
    and "mid_self_attn.proj_out.weight" not in state_dict
):
    state_dict = dict(state_dict)
    state_dict["mid_self_attn.proj_out.weight"] = state_dict.pop("mid_self_attn.proj.weight")
    state_dict["mid_self_attn.proj_out.bias"] = state_dict.pop("mid_self_attn.proj.bias")
elif (
    "mid_self_attn.proj_out.weight" in state_dict
    and "mid_self_attn.proj.weight" in model_keys
    and "mid_self_attn.proj.weight" not in state_dict
):
    state_dict = dict(state_dict)
    state_dict["mid_self_attn.proj.weight"] = state_dict.pop("mid_self_attn.proj_out.weight")
    state_dict["mid_self_attn.proj.bias"] = state_dict.pop("mid_self_attn.proj_out.bias")

ret_unet.load_state_dict(state_dict, strict=True)
ret_unet.eval()
print("Loaded ret_diffusion from epoch:", checkpoint["epoch"])
print("Best monitored score:", checkpoint["best_score"])

train_loader = make_train_loader()
retrieval_bank = RetrievalBank(device=device)
retrieval_bank.build(vae=vae, dataloader=train_loader, device=device, max_items=1000)


## Mel-to-Waveform Converter
This section creates the Griffin-Lim mel-to-waveform converter used throughout the notebook. It converts predicted mel spectrograms back into approximate audio waveforms using the spectrogram configuration from the repo. The converter is used both for full-reconstruction previews and for saving original and predicted gap clips.


In [ ]:
# CELL 12 - Mel2Waveform converter
converter = Mel2Waveform(config_path=str(MEL_CONFIG_PATH), griffinlim_iters=GRIFFINLIM_ITERS)


## Reconstruction Helper
`reconstruct_from_debug()` iterates over every sample in the test split and runs the supplied model inference function on each batch. For each sample, it uses `recording_idx` and `clip_idx` to find the matching reference debug WAV and metadata on HuggingFace Hub. The function saves comparison audio, keeps the original and predicted gap mel spectrograms in a result dictionary, and returns a list of all matched samples.


In [ ]:
# CELL 13 - reconstruct_from_debug function

from huggingface_hub import list_repo_files

_DEBUG_PREFIX_CACHE = {}

def _clip_folder_to_idx(clip_folder):
    # Debug folders are named like clip_0004. Keep only the numeric suffix.
    digits = "".join(ch for ch in str(clip_folder) if ch.isdigit())
    return int(digits) if digits else None


def _build_debug_prefix_index(hf_repo_id, variant, split):
    cache_key = (hf_repo_id, variant, split)
    if cache_key in _DEBUG_PREFIX_CACHE:
        return _DEBUG_PREFIX_CACHE[cache_key]

    root = f"debug/{variant}/{split}/"
    files = list_repo_files(repo_id=hf_repo_id, repo_type="dataset")
    entries = []

    for name in files:
        if not name.startswith(root) or not name.endswith("/metadata.json"):
            continue

        parts = name.split("/")
        if len(parts) < 6:
            continue

        gap_folder = parts[3]
        clip_folder = parts[4]
        clip_folder_idx = _clip_folder_to_idx(clip_folder)
        prefix = "/".join(parts[:-1])

        try:
            gap_seconds_from_folder = float(gap_folder)
        except ValueError:
            gap_seconds_from_folder = None

        try:
            local_meta = hf_hub_download(
                repo_id=hf_repo_id,
                filename=name,
                repo_type="dataset",
            )
            with open(local_meta) as f:
                meta = json.load(f)
        except Exception as exc:
            print(f"Skipping unreadable debug metadata: {name} ({type(exc).__name__}: {exc})")
            continue

        if "recording_idx" not in meta or "clip_idx" not in meta:
            print(f"Skipping debug metadata without recording_idx/clip_idx: {name}")
            continue

        try:
            recording_idx = int(meta["recording_idx"])
            clip_idx = int(meta["clip_idx"])
        except (TypeError, ValueError):
            print(f"Skipping debug metadata with invalid recording_idx/clip_idx: {name}")
            continue

        gap_seconds = meta.get("gap_seconds", gap_seconds_from_folder)
        try:
            gap_seconds = float(gap_seconds)
        except (TypeError, ValueError):
            gap_seconds = gap_seconds_from_folder

        entries.append({
            "prefix": prefix,
            "metadata_path": name,
            "meta": meta,
            "recording_idx": recording_idx,
            "clip_idx": clip_idx,
            "gap_folder": gap_folder,
            "gap_seconds": gap_seconds,
            "clip_folder": clip_folder,
            "clip_folder_idx": clip_folder_idx,
        })

    by_recording_clip = defaultdict(list)
    for entry in entries:
        by_recording_clip[(entry["recording_idx"], entry["clip_idx"])].append(entry)

    index = {"entries": entries, "by_recording_clip": by_recording_clip}
    _DEBUG_PREFIX_CACHE[cache_key] = index
    print(f"Indexed {len(entries)} debug metadata files under {root}")
    return index


def resolve_debug_entry_by_metadata(
    hf_repo_id,
    variant,
    split,
    recording_idx,
    clip_idx,
    gap_seconds=None,
    audio_filename=None,
):
    index = _build_debug_prefix_index(hf_repo_id, variant, split)
    key = (int(recording_idx), int(clip_idx))
    candidates = index["by_recording_clip"].get(key, [])

    if not candidates:
        return None

    if audio_filename is not None:
        audio_matches = []
        for candidate in candidates:
            meta_audio = candidate["meta"].get("audio_filename")
            if meta_audio is None:
                continue
            if (
                str(meta_audio) == str(audio_filename)
                or Path(str(meta_audio)).name == Path(str(audio_filename)).name
            ):
                audio_matches.append(candidate)
        if audio_matches:
            candidates = audio_matches

    if gap_seconds is not None:
        try:
            target_gap = float(gap_seconds)
            candidates_with_gap = [c for c in candidates if c["gap_seconds"] is not None]
            if candidates_with_gap:
                return min(candidates_with_gap, key=lambda c: abs(c["gap_seconds"] - target_gap))
        except (TypeError, ValueError):
            pass

    return candidates[0]


def _pad_or_clip_audio(audio_np, target_len):
    if len(audio_np) < target_len:
        return np.pad(audio_np, (0, target_len - len(audio_np)))
    return audio_np[:target_len]


def _sample_id(recording_idx, clip_idx):
    return f"rec{int(recording_idx):03d}_clip{int(clip_idx):02d}"


def reconstruct_from_debug(
    infer_fn,
    model_name,
    dataloader,
    device,
    converter,
    hf_repo_id=HF_DATASET_REPO,
    variant=VARIANT,
    split="test",
    gdrive_out=GDRIVE_OUT,
    display_audio=True,
    max_missing_debug_prints=10,
    max_examples=None,
):
    sample_results = []
    missing_debug_prints = 0
    displayed_first_match = False
    matched_prefixes = set()

    debug_index = _build_debug_prefix_index(hf_repo_id, variant, split)
    target_debug_count = len(debug_index["entries"])
    if max_examples is not None:
        target_debug_count = min(target_debug_count, int(max_examples))

    if target_debug_count == 0:
        print(f"No debug metadata files were indexed for debug/{variant}/{split}.")
        return sample_results

    for batch in dataloader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        with torch.no_grad():
            pred = infer_fn(x, mask)

        # Merge in spectrogram space for full-clip comparison; gap WAVs use raw gap slices.
        merged = x * (1.0 - mask) + pred * mask

        for i in range(x.shape[0]):
            recording_idx = int(batch["recording_idx"][i])
            clip_idx = int(batch["clip_idx"][i])
            sample_id = _sample_id(recording_idx, clip_idx)
            audio_filename = batch["audio_filename"][i] if "audio_filename" in batch else None
            gap_seconds = batch["gap_seconds"][i] if "gap_seconds" in batch else None

            debug_entry = resolve_debug_entry_by_metadata(
                hf_repo_id=hf_repo_id,
                variant=variant,
                split=split,
                recording_idx=recording_idx,
                clip_idx=clip_idx,
                gap_seconds=gap_seconds,
                audio_filename=audio_filename,
            )

            if debug_entry is None:
                if missing_debug_prints < max_missing_debug_prints:
                    print(
                        f"No debug metadata match for "
                        f"recording_idx={recording_idx}, clip_idx={clip_idx}, "
                        f"gap_seconds={gap_seconds}, audio_filename={audio_filename}"
                    )
                    missing_debug_prints += 1
                continue

            hf_prefix = debug_entry["prefix"]
            if hf_prefix in matched_prefixes:
                continue

            gap_str = hf_prefix.split("/")[3]
            clip_str = hf_prefix.split("/")[4]

            try:
                path_clean = hf_hub_download(repo_id=hf_repo_id, filename=f"{hf_prefix}/clean_full.wav", repo_type="dataset")
            except Exception as exc:
                if missing_debug_prints < max_missing_debug_prints:
                    print(f"Skipping missing debug files: {hf_prefix}")
                    print(f"  {type(exc).__name__}: {exc}")
                    missing_debug_prints += 1
                continue

            meta = debug_entry["meta"]

            sr = int(meta["sample_rate"])
            if "mask_start_sample" in meta and "mask_end_sample" in meta:
                gap_start_sample = int(meta["mask_start_sample"])
                gap_end_sample = int(meta["mask_end_sample"])
            else:
                hop_length = int(meta.get("hop_length", HOP_LENGTH))
                gap_start_sample = int(meta["mask_start_frame"]) * hop_length
                gap_end_sample = int(meta["mask_end_frame"]) * hop_length
            gap_len = gap_end_sample - gap_start_sample

            clean_wav, _ = torchaudio.load(path_clean)
            clean_full_hf = clean_wav.numpy().squeeze()
            target_len = clean_wav.shape[-1]

            start_f = int(meta.get("mask_start_frame", batch["mask_start_frame"][i]))
            end_f = int(meta.get("mask_end_frame", batch["mask_end_frame"][i]))
            original_gap_spec = y[i, 0, :, start_f:end_f].detach().cpu().numpy()
            pred_gap_spec = pred[i, 0, :, start_f:end_f].detach().cpu().numpy()

            clean_fullmel = converter.to_waveform_(y[i, 0].detach().cpu().numpy())
            clean_fullmel = _pad_or_clip_audio(clean_fullmel, target_len)
            ground_truth_gap = clean_full_hf[gap_start_sample:gap_end_sample]
            predicted_gap = converter.to_waveform_(pred_gap_spec)
            predicted_gap = _pad_or_clip_audio(predicted_gap, gap_len)

            out_dir = gdrive_out / model_name / variant / split / gap_str / clip_str
            out_dir.mkdir(parents=True, exist_ok=True)

            saved_paths = {
                "ground_truth_full_hf": out_dir / "ground_truth_full_hf.wav",
                "clean_fullmel_griffin_lim": out_dir / "clean_fullmel_griffin_lim.wav",
                "ground_truth_gap_hf": out_dir / "ground_truth_gap_hf.wav",
                "predicted_gap_griffin_lim": out_dir / "predicted_gap_griffin_lim.wav",
            }

            sf.write(saved_paths["ground_truth_full_hf"], clean_full_hf, sr)
            sf.write(saved_paths["clean_fullmel_griffin_lim"], clean_fullmel, SAMPLE_RATE)
            sf.write(saved_paths["ground_truth_gap_hf"], ground_truth_gap, sr)
            sf.write(saved_paths["predicted_gap_griffin_lim"], predicted_gap, SAMPLE_RATE)

            compare_url = f"https://huggingface.co/datasets/{hf_repo_id}/resolve/main/{hf_prefix}/clean_full.wav"
            with open(out_dir / "compare_with_debug.txt", "w") as f:
                f.write(f"HF debug prefix:\n{hf_prefix}\n\n")
                f.write(f"Compare against debug clean_full.wav:\n{compare_url}\n\n")
                f.write("Saved files:\n")
                for label, path in saved_paths.items():
                    f.write(f"{label}: {path.name}\n")

            print(f"[{model_name}] saved {sample_id} -> {out_dir}")
            for label, path in saved_paths.items():
                print(f"  {label}: {path}")

            result = {
                "model_name": model_name,
                "recording_idx": recording_idx,
                "clip_idx": clip_idx,
                "sample_id": sample_id,
                "audio_filename": audio_filename,
                "hf_prefix": hf_prefix,
                "output_dir": out_dir,
                "saved_paths": saved_paths,
                "mask_start_frame": start_f,
                "mask_end_frame": end_f,
                "original_gap_mel": original_gap_spec,
                "pred_gap_mel": pred_gap_spec,
            }
            sample_results.append(result)
            matched_prefixes.add(hf_prefix)

            if display_audio and not displayed_first_match:
                print(f"\nShowing first matched sample only: {sample_id}")
                display_items = [
                    ("Ground truth full clip from HuggingFace", clean_full_hf, sr),
                    ("Clean full mel reconstructed with Griffin-Lim", clean_fullmel, SAMPLE_RATE),
                    ("Ground truth gap clip from HuggingFace", ground_truth_gap, sr),
                    ("Predicted gap clip reconstructed with Griffin-Lim", predicted_gap, SAMPLE_RATE),
                ]
                for label, audio_np, audio_sr in display_items:
                    print(f"{label}:")
                    display(Audio(audio_np, rate=audio_sr))
                displayed_first_match = True

            if len(sample_results) >= target_debug_count:
                print(f"[{model_name}] Matched {len(sample_results)} debug folders; stopping scan.")
                break

        if len(sample_results) >= target_debug_count:
            break

    if len(sample_results) == 0:
        print("No examples were saved because no matching debug WAV/metadata files were found for the parquet rows.")
        print("Check the printed metadata match diagnostics.")
    print(f"\n[{model_name}] Saved {len(sample_results)} matched examples -> {gdrive_out / model_name}")
    return sample_results


## Inference: Run All Models
Each cell below re-creates a fresh dataloader before running inference for one model. This is required because the HuggingFace streaming `IterableDataset` is a one-pass stream, so a consumed loader cannot be reused for the next model. Each model then calls `reconstruct_from_debug()` over all test samples and saves per-sample original and predicted gap WAV files.


In [ ]:
# CELL 14 - Run CNN-AE

# fresh loader - streaming iterator starts from the beginning
test_loader = make_test_loader()

print("Running CNN-AE...")
cnn_ae_results = reconstruct_from_debug(
    infer_fn=lambda x, m: cnn_ae(x, m),
    model_name="cnn_ae",
    dataloader=test_loader,
    device=device,
    converter=converter,
)

# NOTE ON infer_fn FOR CNN-AE: CNNAutoencoder.forward returns x + mask * raw_pred
# internally (predict_residual=True). reconstruct_from_debug then computes
# x*(1-mask) + pred*mask. With a binary mask this simplifies to
# x*(1-mask) + x*mask + mask*raw_pred = x + mask*raw_pred - the correct merged
# result. No special handling needed.


In [ ]:
# Plot CNN-AE inference spectrograms
for result in cnn_ae_results:
    sample_id = result["sample_id"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    orig_power = np.expm1(result["original_gap_mel"])
    pred_power = np.expm1(result["pred_gap_mel"])

    orig_power = np.maximum(orig_power, 0.0)
    pred_power = np.maximum(pred_power, 0.0)

    ref_power = max(orig_power.max(), pred_power.max(), 1e-8)

    orig_db = librosa.power_to_db(orig_power, ref=ref_power)
    pred_db = librosa.power_to_db(pred_power, ref=ref_power)

    vmin = -80
    vmax = 0

    img0 = librosa.display.specshow(
        orig_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[0],
    )
    axes[0].set_title("Original Gap")
    plt.colorbar(img0, ax=axes[0], format="%+2.0f dB")

    img1 = librosa.display.specshow(
        pred_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[1],
    )
    axes[1].set_title("Predicted Gap (cnn_ae)")
    plt.colorbar(img1, ax=axes[1], format="%+2.0f dB")

    fig.suptitle(sample_id)
    plt.tight_layout()
    plt.show()

In [ ]:
# CELL 15 - Run U-Net

# fresh loader - mandatory, streaming iterator must restart
test_loader = make_test_loader()

print("Running U-Net...")
unet_results = reconstruct_from_debug(
    infer_fn=lambda x, m: unet_model(x, m),
    model_name="unet",
    dataloader=test_loader,
    device=device,
    converter=converter,
)

# NOTE ON infer_fn FOR U-Net: UNet.forward returns the raw prediction from the
# output convolution only - no residual merge inside the model. The merge is
# applied correctly in reconstruct_from_debug.


In [ ]:
# Plot U-Net inference spectrograms
for result in unet_results:
    sample_id = result["sample_id"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    orig_power = np.expm1(result["original_gap_mel"])
    pred_power = np.expm1(result["pred_gap_mel"])

    orig_power = np.maximum(orig_power, 0.0)
    pred_power = np.maximum(pred_power, 0.0)

    ref_power = max(orig_power.max(), pred_power.max(), 1e-8)

    orig_db = librosa.power_to_db(orig_power, ref=ref_power)
    pred_db = librosa.power_to_db(pred_power, ref=ref_power)

    vmin = -80
    vmax = 0

    img0 = librosa.display.specshow(
        orig_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[0],
    )
    axes[0].set_title("Original Gap")
    plt.colorbar(img0, ax=axes[0], format="%+2.0f dB")

    img1 = librosa.display.specshow(
        pred_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[1],
    )
    axes[1].set_title("Predicted Gap (unet)")
    plt.colorbar(img1, ax=axes[1], format="%+2.0f dB")

    fig.suptitle(sample_id)
    plt.tight_layout()
    plt.show()

In [ ]:
# CELL 16 - Run Base Diffusion

# fresh loader
test_loader = make_test_loader()

print("Running Base Diffusion...")
base_diffusion_results = reconstruct_from_debug(
    infer_fn=lambda x, m: infer_base_diffusion(
        vae, diff_unet, diff_schedule, x, m, device,
        num_steps=200, use_self_conditioning=True,
    ),
    model_name="base_diffusion",
    dataloader=test_loader,
    device=device,
    converter=converter,
)

# NOTE ON infer_fn FOR BASE DIFFUSION: infer_base_diffusion returns the raw VAE
# decoded prediction - no merge. reconstruct_from_debug applies the merge once.


In [ ]:
# Plot Base Diffusion inference spectrograms
for result in base_diffusion_results:
    sample_id = result["sample_id"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    orig_power = np.expm1(result["original_gap_mel"])
    pred_power = np.expm1(result["pred_gap_mel"])

    orig_power = np.maximum(orig_power, 0.0)
    pred_power = np.maximum(pred_power, 0.0)

    ref_power = max(orig_power.max(), pred_power.max(), 1e-8)

    orig_db = librosa.power_to_db(orig_power, ref=ref_power)
    pred_db = librosa.power_to_db(pred_power, ref=ref_power)

    vmin = -80
    vmax = 0

    img0 = librosa.display.specshow(
        orig_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[0],
    )
    axes[0].set_title("Original Gap")
    plt.colorbar(img0, ax=axes[0], format="%+2.0f dB")

    img1 = librosa.display.specshow(
        pred_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[1],
    )
    axes[1].set_title("Predicted Gap (base_diffusion)")
    plt.colorbar(img1, ax=axes[1], format="%+2.0f dB")

    fig.suptitle(sample_id)
    plt.tight_layout()
    plt.show()

In [ ]:
# CELL 17 - Run Retrieval-Conditioned Diffusion

# fresh loader
test_loader = make_test_loader()

print("Running Retrieval-Conditioned Diffusion...")
ret_diffusion_results = reconstruct_from_debug(
    infer_fn=lambda x, m: infer_ret_diffusion(
        vae, ret_unet, retrieval_bank, diff_schedule,
        x, m, device, num_steps=200,
        top_k=5, max_keep=3,
    ),
    model_name="ret_diffusion",
    dataloader=test_loader,
    device=device,
    converter=converter,
)

# NOTE ON infer_fn FOR RET DIFFUSION: infer_ret_diffusion returns the raw VAE
# decoded prediction - no merge. reconstruct_from_debug applies the merge once.


In [ ]:
# Plot Retrieval-Conditioned Diffusion inference spectrograms
for result in ret_diffusion_results:
    sample_id = result["sample_id"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    orig_power = np.expm1(result["original_gap_mel"])
    pred_power = np.expm1(result["pred_gap_mel"])

    orig_power = np.maximum(orig_power, 0.0)
    pred_power = np.maximum(pred_power, 0.0)

    ref_power = max(orig_power.max(), pred_power.max(), 1e-8)

    orig_db = librosa.power_to_db(orig_power, ref=ref_power)
    pred_db = librosa.power_to_db(pred_power, ref=ref_power)

    vmin = -80
    vmax = 0

    img0 = librosa.display.specshow(
        orig_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[0],
    )
    axes[0].set_title("Original Gap")
    plt.colorbar(img0, ax=axes[0], format="%+2.0f dB")

    img1 = librosa.display.specshow(
        pred_db,
        y_axis="mel",
        x_axis="time",
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        vmin=vmin,
        vmax=vmax,
        ax=axes[1],
    )
    axes[1].set_title("Predicted Gap (ret_diffusion)")
    plt.colorbar(img1, ax=axes[1], format="%+2.0f dB")

    fig.suptitle(sample_id)
    plt.tight_layout()
    plt.show()

## Output Summary
`print_folder_tree()` prints a directory listing of all saved results under the configured output folder. This gives a compact view of which model outputs, variants, splits, debug comparisons, and audio files were written. It is useful as a final sanity check after running the full reverse pipeline.


In [ ]:
# CELL 18 - Output summary
def print_folder_tree(root):
    root = Path(root)
    if not root.exists():
        print(f"Missing output folder: {root}")
        return
    for current_root, dirs, files in os.walk(root):
        current_root = Path(current_root)
        level = len(current_root.relative_to(root).parts)
        indent = "  " * level
        print(f"{indent}{current_root.name}/")
        for name in sorted(files):
            print(f"{indent}  {name}")


print_folder_tree(GDRIVE_OUT)

print("\nReminders:")
print("1. To switch to long_gaps change VARIANT = 'long_gaps' in CELL 3 and re-run from CELL 4 onwards.")
print("2. The retrieval checkpoint path is set as ret_ckpt_path in CELL 11e. Update it if the Drive location differs.")
print("3. The retrieval bank was built from training data. For higher-quality retrieval, increase max_items in CELL 11e (uses more RAM and build time).")
